In [1]:
#Starts from here
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression  # LogisticRegression is not used for regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler 
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
from tqdm import tqdm

def remove_low_variance_columns(df, threshold=0.005):
    # df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()
    
    # Identify columns with variance below the threshold
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

def features(df, target_column='Permeability', threshold=0.9):
    correlation_matrix = df.corr()
    
    features_to_drop = set()
    
    for feature in correlation_matrix.columns:
        if feature == target_column:
            continue 
        target_corr = correlation_matrix[target_column][feature]
        
        for other_feature in correlation_matrix.columns:
            if other_feature == feature or other_feature == target_column:
                continue
            
            if abs(correlation_matrix[feature][other_feature]) > threshold:
                other_target_corr = correlation_matrix[target_column][other_feature]

                if abs(other_target_corr) < abs(target_corr):
                    features_to_drop.add(other_feature)
                else:
                    features_to_drop.add(feature)
    selected_features = [col for col in df.columns if col not in features_to_drop and col != target_column]
    
    return selected_features

In [2]:
from tqdm import tqdm
# 2D and 3D descriptors dataframes
df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Descriptors/Train_2d_3d_all_descriptors_MDCK.csv')
df_train = df_desc_train.sort_values(by='ID')
df_train =df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_desc_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_desc_test = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Descriptors/Test_2d_3d_all_descriptors_MDCK.csv')
df_desc_test = df_desc_test.sort_values(by='ID')
df_desc_test =df_desc_test.dropna()
df_desc_test =  df_desc_test[df_desc_train.columns]


# Fingerprints
df_fp_train = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Fingerprints/Train/All_fingerprints_train_MDCK.csv')
df_train = df_fp_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_fp_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_fp_test = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Fingerprints/Test/All_fingerprints_test_MDCK.csv')
df_fp_test = df_fp_test.sort_values(by='ID')
df_fp_test = df_fp_test.dropna()
df_fp_test =  df_fp_test[df_fp_train.columns]


#Smiles Embeddings
df_emb_train = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Embeddings/Train_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_mdck.csv')
df_train = df_emb_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_emb_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_emb_test = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Embeddings/Test_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_mdck.csv')
df_emb_test = df_emb_test.sort_values(by='ID')
df_emb_test = df_emb_test.dropna()
df_emb_test =  df_emb_test[df_emb_train.columns]

#ATomic features
df_atomic_train = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Atomic/Train_all_atomic_desc_MDCK.csv')
df_train = df_atomic_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_atomic_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
# df_atomic_train =pd.concat( [df_train['SMILES'], df_train.select_dtypes(include=['number'])], axis=1)
df_atomic_test = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Atomic/Test_all_atomic_desc_MDCK.csv')
df_atomic_test = df_atomic_test.sort_values(by='ID')
df_atomic_test = df_atomic_test.dropna()
df_atomic_test =  df_atomic_test[df_atomic_train.columns]


print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Loading completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
df_fp_test = df_fp_test[df_fp_test['ID'].isin(df_desc_test['ID'])]
df_fp_train = df_fp_train[df_fp_train['ID'].isin(df_desc_train['ID'])]

df_emb_test = df_emb_test[df_emb_test['ID'].isin(df_desc_test['ID'])]
df_emb_train = df_emb_train[df_emb_train['ID'].isin(df_desc_train['ID'])]

df_atomic_test = df_atomic_test[df_atomic_test['ID'].isin(df_desc_test['ID'])]
df_atomic_train = df_atomic_train[df_atomic_train['ID'].isin(df_desc_train['ID'])]
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Processing completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train.shape)
print(df_desc_test.shape)
print(df_fp_train.shape)
print(df_fp_test.shape)
print(df_emb_train.shape)
print(df_emb_test.shape)
print(df_atomic_train.shape)
print(df_atomic_test.shape)

XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Loading completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Processing completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
(51, 175)
(13, 175)
(51, 91)
(13, 91)
(51, 573)
(13, 573)
(51, 8)
(13, 8)


In [3]:
merge_keys = ['ID', 'SMILES', 'Permeability']

merged_train = df_desc_train.merge(df_fp_train, on=merge_keys)
merged_train = merged_train.merge(df_emb_train, on=merge_keys)
merged_train = merged_train.merge(df_atomic_train, on=merge_keys)

merged_test = df_desc_test.merge(df_fp_test, on=merge_keys)
merged_test = merged_test.merge(df_emb_test, on=merge_keys)
merged_test = merged_test.merge(df_atomic_test, on=merge_keys)

In [4]:
X_train = merged_train.drop(columns=['ID', 'SMILES']).select_dtypes(include=['number'])
selected_final_features = features(X_train, target_column='Permeability')

train = pd.concat([merged_train[['ID', 'SMILES', 'Permeability']], X_train[selected_final_features]], axis=1)
test = merged_test[train.columns] 

print('selected_final_features', selected_final_features )
print("Final Train shape:", train.shape)
print("Final Test shape:", test.shape)

selected_final_features ['MinAbsEStateIndex', 'SPS', 'FpDensityMorgan1', 'Ipc', 'EState_VSA11', 'AdjacencyMatrix.6', 'AdjacencyMatrix.11', 'AATS.8', 'AATS.11', 'AATS.23', 'AATS.24', 'AATS.31', 'AATS.33', 'AATS.44', 'AATS.96', 'AATS.98', 'ATSC.4', 'ATSC.5', 'ATSC.20', 'ATSC.22', 'ATSC.23', 'ATSC.24', 'ATSC.25', 'ATSC.26', 'ATSC.30', 'ATSC.32', 'ATSC.35', 'ATSC.43', 'ATSC.64', 'ATSC.79', 'ATSC.84', 'ATSC.93', 'ATSC.105', 'ATSC.106', 'AATSC.9', 'AATSC.11', 'AATSC.14', 'AATSC.16', 'AATSC.17', 'AATSC.37', 'AATSC.39', 'AATSC.50', 'AATSC.57', 'GATS.4', 'GATS.12', 'GATS.22', 'GATS.62', 'GATS.86', 'AtomTypeEState.91', 'AtomTypeEState.173', 'AtomTypeEState.187', 'AtomTypeEState.252', 'AtomTypeEState.260', 'AtomTypeEState.271', 'ComplementaryIC.4', 'ModifiedIC.3', 'MolecularId.5', 'RingCount.52', 'ALogP', 'AATS3v', 'AATS3i', 'AATS4i', 'AATS7i', 'ATSC7c', 'ATSC8e', 'ATSC5p', 'ATSC6p', 'ATSC3i', 'ATSC5i', 'ATSC1s', 'ATSC2s', 'AATSC5v', 'AATSC8v', 'VR3_Dzv', 'VE3_Dzp', 'SM1_Dzs', 'VR2_Dzs', 'BCUTp-1

In [5]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -4.0)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -4.0)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df



In [6]:
X_train = train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test[X_train.columns]
y_test = test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 824)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 824)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.130490 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2844
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 194
[LightGBM] [Info] Start training from score -5.596983
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3631,0.4857,0.6026,0.3100,0.5582,0.5776,0.4598,0.5331,0.6781,0.3571,0.6445,0.5372
DecisionTreeRegressor,0.7639,0.6019,0.8740,-0.4519,0.2307,0.3723,0.2954,0.3949,0.5435,0.5869,0.8118,0.8540
RandomForestRegressor,0.3942,0.4969,0.6279,0.2508,0.5036,0.5476,0.4073,0.5197,0.6382,0.4305,0.7204,0.5510
GradientBoostingRegressor,0.4819,0.5129,0.6942,0.0842,0.4136,0.4210,0.3657,0.4797,0.6047,0.4886,0.7547,0.5950
AdaBoostRegressor,0.4093,0.5035,0.6397,0.2222,0.5043,0.5290,0.3968,0.4889,0.6300,0.4451,0.7130,0.6860
XGBRegressor,0.4563,0.5207,0.6755,0.1327,0.4329,0.4476,0.3227,0.4525,0.5681,0.5487,0.8195,0.8209
ExtraTreesRegressor,0.4109,0.5152,0.6410,0.2190,0.4757,0.4997,0.3978,0.4944,0.6307,0.4437,0.6992,0.5455
LinearRegression,0.7142,0.6897,0.8451,-0.3573,0.3499,0.3566,0.2715,0.3656,0.5211,0.6204,0.7970,0.8650
KNeighborsRegressor,0.3805,0.4852,0.6168,0.2769,0.5437,0.5865,0.3310,0.4426,0.5753,0.5372,0.7773,0.6997
SVR,0.3992,0.5217,0.6318,0.2412,0.4917,0.5065,0.3576,0.4699,0.5980,0.5000,0.8049,0.7879


In [7]:
result_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/combined_features/combined_features_MDCK.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/combined_features/prediction_data_combined_features_MDCK.csv')

In [8]:
X = train.drop(columns=['ID', 'SMILES', 'Permeability'])
y = train['Permeability']

rf = RandomForestRegressor(n_estimators=100, random_state=101, n_jobs=-1)
rf.fit(X, y)

importances = rf.feature_importances_
feature_names = X.columns


In [9]:
#Top 10 features
n = 10  
top_10_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_10_features = feature_names[top_10_indices].tolist() 

# Output the list
print("Top", 10, "features:\n")
print(top_10_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_10_features]], axis=1)
test_df = test[train.columns] 

Top 10 features:

['x_fine_emb_MFXL590', 'x_fine_emb_MFXL448', 'x_fine_emb_MFXL613', 'x_fine_emb_MFXL45', 'AATSC.17', 'x_fine_emb_MFXL226', 'x_fine_emb_MFXL386', 'x_fine_emb_MFXL12', 'x_fine_emb_MFXL146', 'x_fine_emb_MFXL366']


In [10]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 10)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 10)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.135496 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 75
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 5
[LightGBM] [Info] Start training from score -5.596983
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3783,0.4776,0.6151,0.2810,0.5394,0.5948,0.4558,0.5721,0.6751,0.3627,0.6528,0.5096
DecisionTreeRegressor,0.4777,0.5487,0.6912,0.0920,0.5328,0.5811,0.4004,0.5007,0.6328,0.4401,0.6895,0.6749
RandomForestRegressor,0.3110,0.4469,0.5577,0.4089,0.6440,0.6965,0.4705,0.5616,0.6859,0.3421,0.5926,0.4325
GradientBoostingRegressor,0.3341,0.4468,0.5780,0.3650,0.6191,0.6663,0.4200,0.5193,0.6480,0.4128,0.6575,0.5482
AdaBoostRegressor,0.3046,0.4195,0.5519,0.4211,0.6556,0.6762,0.4560,0.5263,0.6753,0.3623,0.6073,0.4628
XGBRegressor,0.4079,0.4886,0.6387,0.2248,0.5096,0.5192,0.4118,0.5412,0.6417,0.4242,0.6839,0.6033
ExtraTreesRegressor,0.3576,0.4917,0.5980,0.3204,0.5732,0.5910,0.4427,0.5224,0.6654,0.3809,0.6239,0.4242
LinearRegression,0.4488,0.5086,0.6699,0.1470,0.4949,0.5529,0.4540,0.5191,0.6738,0.3651,0.6170,0.4904
KNeighborsRegressor,0.3819,0.5023,0.6180,0.2742,0.5864,0.6378,0.5824,0.6168,0.7631,0.1856,0.4868,0.3021
SVR,0.3594,0.5041,0.5995,0.3170,0.5673,0.5734,0.4959,0.5396,0.7042,0.3066,0.5540,0.4298


In [11]:
result_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/combined_features/combined_top_10_features_MDCK.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/combined_features/prediction_data_combined_top_10_features_MDCK.csv')

In [12]:
#Top 20 features
n = 20  
top_20_indices = importances.argsort()[::-1][:n]  
top_20_features = feature_names[top_20_indices].tolist()  # convert to list

# Output the list
print("Top", 20, "features:\n")
print(top_20_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_20_features]], axis=1)
test_df = test[train.columns] 

Top 20 features:

['x_fine_emb_MFXL590', 'x_fine_emb_MFXL448', 'x_fine_emb_MFXL613', 'x_fine_emb_MFXL45', 'AATSC.17', 'x_fine_emb_MFXL226', 'x_fine_emb_MFXL386', 'x_fine_emb_MFXL12', 'x_fine_emb_MFXL146', 'x_fine_emb_MFXL366', 'x_fine_emb_MFXL514', 'DPSA-3', 'x_fine_emb_MFXL95', 'x_fine_emb_MFXL762', 'x_fine_emb_MFXL511', 'x_fine_emb_MFXL502', 'x_fine_emb_MFXL39', 'AtomTypeEState.187', 'x_fine_emb_MFXL125', 'AtomTypeEState.260']


In [13]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 20)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 20)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.122972 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 120
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 8
[LightGBM] [Info] Start training from score -5.596983
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3570,0.4722,0.5975,0.3214,0.5699,0.6013,0.4920,0.5737,0.7014,0.3121,0.5802,0.4601
DecisionTreeRegressor,0.5592,0.5692,0.7478,-0.0627,0.4753,0.5682,0.3542,0.4358,0.5951,0.5047,0.7473,0.7548
RandomForestRegressor,0.3099,0.4392,0.5567,0.4110,0.6425,0.6779,0.4406,0.5385,0.6638,0.3839,0.6380,0.5923
GradientBoostingRegressor,0.3425,0.4612,0.5852,0.3491,0.6059,0.6599,0.3801,0.4963,0.6165,0.4686,0.7230,0.5703
AdaBoostRegressor,0.3120,0.4380,0.5586,0.4070,0.6476,0.6506,0.4365,0.5103,0.6607,0.3896,0.6364,0.5069
XGBRegressor,0.3757,0.4853,0.6129,0.2860,0.5569,0.5676,0.4317,0.5443,0.6571,0.3963,0.6564,0.5730
ExtraTreesRegressor,0.3242,0.4624,0.5694,0.3839,0.6208,0.6444,0.4148,0.5035,0.6440,0.4200,0.6627,0.5372
LinearRegression,0.7147,0.6855,0.8454,-0.3582,0.4197,0.4534,0.6007,0.6334,0.7751,0.1599,0.4651,0.4050
KNeighborsRegressor,0.3508,0.4635,0.5923,0.3332,0.6325,0.6956,0.5816,0.5613,0.7626,0.1867,0.4998,0.5565
SVR,0.3567,0.5137,0.5972,0.3221,0.5710,0.5641,0.4587,0.5106,0.6773,0.3586,0.6096,0.5069


In [14]:
result_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/combined_features/combined_top_20_features_MDCK.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/combined_features/prediction_data_combined_top_20_features_MDCK.csv')

In [15]:
#Top 50 features
n = 50  
top_50_indices = importances.argsort()[::-1][:n] 
top_50_features = feature_names[top_50_indices].tolist()  # convert to list

# Output the list
print("Top", 50, "features:\n")
print(top_50_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_50_features]], axis=1)
test_df = test[train.columns] 

Top 50 features:

['x_fine_emb_MFXL590', 'x_fine_emb_MFXL448', 'x_fine_emb_MFXL613', 'x_fine_emb_MFXL45', 'AATSC.17', 'x_fine_emb_MFXL226', 'x_fine_emb_MFXL386', 'x_fine_emb_MFXL12', 'x_fine_emb_MFXL146', 'x_fine_emb_MFXL366', 'x_fine_emb_MFXL514', 'DPSA-3', 'x_fine_emb_MFXL95', 'x_fine_emb_MFXL762', 'x_fine_emb_MFXL511', 'x_fine_emb_MFXL502', 'x_fine_emb_MFXL39', 'AtomTypeEState.187', 'x_fine_emb_MFXL125', 'AtomTypeEState.260', 'TDB7i', 'GATS.4', 'x_fine_emb_MFXL574', 'TDB9s', 'x_fine_emb_MFXL471', 'ATSC8e', 'x_fine_emb_MFXL766', 'x_fine_emb_MFXL434', 'x_fine_emb_MFXL684', 'x_fine_emb_MFXL566', 'x_fine_emb_MFXL249', 'AATS7i', 'x_fine_emb_MFXL721', 'x_fine_emb_MFXL504', 'x_fine_emb_MFXL751', 'x_fine_emb_MFXL184', 'x_fine_emb_MFXL305', 'x_fine_emb_MFXL673', 'TDB10m', 'x_fine_emb_MFXL147', 'maxHBint9', 'x_fine_emb_MFXL8', 'minHBint5', 'x_fine_emb_MFXL407', 'x_fine_emb_MFXL317', 'x_fine_emb_MFXL193', 'x_fine_emb_MFXL740', 'AATS.33', 'x_fine_emb_MFXL394', 'x_fine_emb_MFXL181']


In [16]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 50)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 50)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.132792 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 240
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 16
[LightGBM] [Info] Start training from score -5.596983
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-1.8290772749360746


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3345,0.4872,0.5784,0.3642,0.6041,0.6342,0.4220,0.5255,0.6496,0.4099,0.6659,0.5592
DecisionTreeRegressor,0.7404,0.6312,0.8605,-0.4072,0.3124,0.3645,0.4509,0.4688,0.6715,0.3695,0.6459,0.6887
RandomForestRegressor,0.3346,0.4627,0.5785,0.3640,0.6040,0.6415,0.4045,0.4980,0.6360,0.4344,0.7080,0.6006
GradientBoostingRegressor,0.3748,0.4767,0.6122,0.2877,0.5512,0.5757,0.3784,0.4543,0.6152,0.4708,0.7223,0.6722
AdaBoostRegressor,0.2902,0.4258,0.5387,0.4486,0.6740,0.7006,0.3900,0.4644,0.6245,0.4546,0.7259,0.6391
XGBRegressor,0.3966,0.5048,0.6297,0.2463,0.5175,0.5444,0.3923,0.4856,0.6264,0.4514,0.7261,0.7025
ExtraTreesRegressor,0.3712,0.4979,0.6092,0.2946,0.5493,0.5868,0.4162,0.5022,0.6451,0.4180,0.6661,0.4849
LinearRegression,7.2640,2.1341,2.6952,-12.8055,-0.1577,-0.1162,1.6927,1.0787,1.3011,-1.3670,0.1142,0.1405
KNeighborsRegressor,0.3508,0.4661,0.5923,0.3333,0.6301,0.6974,0.4101,0.4983,0.6404,0.4265,0.6563,0.6428
SVR,0.3173,0.4737,0.5633,0.3969,0.6320,0.6321,0.3274,0.4278,0.5722,0.5422,0.8286,0.7989


In [17]:
result_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/combined_features/combined_top_50_features_MDCK.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/combined_features/prediction_data_combined_top_50_features_MDCK.csv')

In [18]:
#Top 100 features
n = 100  
top_100_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_100_features = feature_names[top_100_indices].tolist()  # convert to list

# Output the list
print("Top", 100, "features:\n")
print(top_100_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_100_features]], axis=1)
test_df = test[train.columns] 

Top 100 features:

['x_fine_emb_MFXL590', 'x_fine_emb_MFXL448', 'x_fine_emb_MFXL613', 'x_fine_emb_MFXL45', 'AATSC.17', 'x_fine_emb_MFXL226', 'x_fine_emb_MFXL386', 'x_fine_emb_MFXL12', 'x_fine_emb_MFXL146', 'x_fine_emb_MFXL366', 'x_fine_emb_MFXL514', 'DPSA-3', 'x_fine_emb_MFXL95', 'x_fine_emb_MFXL762', 'x_fine_emb_MFXL511', 'x_fine_emb_MFXL502', 'x_fine_emb_MFXL39', 'AtomTypeEState.187', 'x_fine_emb_MFXL125', 'AtomTypeEState.260', 'TDB7i', 'GATS.4', 'x_fine_emb_MFXL574', 'TDB9s', 'x_fine_emb_MFXL471', 'ATSC8e', 'x_fine_emb_MFXL766', 'x_fine_emb_MFXL434', 'x_fine_emb_MFXL684', 'x_fine_emb_MFXL566', 'x_fine_emb_MFXL249', 'AATS7i', 'x_fine_emb_MFXL721', 'x_fine_emb_MFXL504', 'x_fine_emb_MFXL751', 'x_fine_emb_MFXL184', 'x_fine_emb_MFXL305', 'x_fine_emb_MFXL673', 'TDB10m', 'x_fine_emb_MFXL147', 'maxHBint9', 'x_fine_emb_MFXL8', 'minHBint5', 'x_fine_emb_MFXL407', 'x_fine_emb_MFXL317', 'x_fine_emb_MFXL193', 'x_fine_emb_MFXL740', 'AATS.33', 'x_fine_emb_MFXL394', 'x_fine_emb_MFXL181', 'x_fine_emb

In [19]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 100)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 100)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.091755 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 433
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 29
[LightGBM] [Info] Start training from score -5.596983
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3465,0.4951,0.5887,0.3414,0.5875,0.5917,0.4044,0.5132,0.6359,0.4346,0.6964,0.5317
DecisionTreeRegressor,0.6727,0.5436,0.8202,-0.2785,0.3297,0.4465,0.3759,0.4638,0.6131,0.4744,0.7015,0.7521
RandomForestRegressor,0.3367,0.4616,0.5803,0.3601,0.6013,0.6287,0.3958,0.4954,0.6291,0.4466,0.7161,0.6226
GradientBoostingRegressor,0.4201,0.5032,0.6481,0.2016,0.4881,0.4946,0.3367,0.4396,0.5802,0.5292,0.7741,0.7603
AdaBoostRegressor,0.3512,0.4639,0.5926,0.3326,0.5835,0.5991,0.4072,0.4782,0.6382,0.4305,0.6851,0.6226
XGBRegressor,0.4439,0.5071,0.6663,0.1564,0.4492,0.4817,0.3920,0.4913,0.6261,0.4519,0.7174,0.7163
ExtraTreesRegressor,0.3632,0.4787,0.6026,0.3098,0.5599,0.6020,0.3988,0.4781,0.6315,0.4424,0.6845,0.6171
LinearRegression,1.7184,1.0266,1.3109,-2.2659,0.2550,0.3477,0.5292,0.5568,0.7275,0.2599,0.6426,0.6667
KNeighborsRegressor,0.3198,0.4392,0.5655,0.3922,0.6383,0.6853,0.3360,0.4241,0.5797,0.5301,0.7776,0.7548
SVR,0.3395,0.4806,0.5826,0.3548,0.5989,0.5962,0.3103,0.4183,0.5570,0.5661,0.8439,0.8209


In [20]:
result_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/combined_features/combined_top_100_features_MDCK.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/combined_features/prediction_data_combined_top_100_features_MDCK.csv')

In [21]:
#Top 200 features
n = 200  
top_200_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_200_features = feature_names[top_200_indices].tolist()  # convert to list

# Output the list
print("Top", 200, "features:\n")
print(top_200_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_200_features]], axis=1)
test_df = test[train.columns]

Top 200 features:

['x_fine_emb_MFXL590', 'x_fine_emb_MFXL448', 'x_fine_emb_MFXL613', 'x_fine_emb_MFXL45', 'AATSC.17', 'x_fine_emb_MFXL226', 'x_fine_emb_MFXL386', 'x_fine_emb_MFXL12', 'x_fine_emb_MFXL146', 'x_fine_emb_MFXL366', 'x_fine_emb_MFXL514', 'DPSA-3', 'x_fine_emb_MFXL95', 'x_fine_emb_MFXL762', 'x_fine_emb_MFXL511', 'x_fine_emb_MFXL502', 'x_fine_emb_MFXL39', 'AtomTypeEState.187', 'x_fine_emb_MFXL125', 'AtomTypeEState.260', 'TDB7i', 'GATS.4', 'x_fine_emb_MFXL574', 'TDB9s', 'x_fine_emb_MFXL471', 'ATSC8e', 'x_fine_emb_MFXL766', 'x_fine_emb_MFXL434', 'x_fine_emb_MFXL684', 'x_fine_emb_MFXL566', 'x_fine_emb_MFXL249', 'AATS7i', 'x_fine_emb_MFXL721', 'x_fine_emb_MFXL504', 'x_fine_emb_MFXL751', 'x_fine_emb_MFXL184', 'x_fine_emb_MFXL305', 'x_fine_emb_MFXL673', 'TDB10m', 'x_fine_emb_MFXL147', 'maxHBint9', 'x_fine_emb_MFXL8', 'minHBint5', 'x_fine_emb_MFXL407', 'x_fine_emb_MFXL317', 'x_fine_emb_MFXL193', 'x_fine_emb_MFXL740', 'AATS.33', 'x_fine_emb_MFXL394', 'x_fine_emb_MFXL181', 'x_fine_emb

In [22]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 200)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 200)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.091135 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 761
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 51
[LightGBM] [Info] Start training from score -5.596983
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3418,0.4675,0.5846,0.3504,0.5971,0.6035,0.4300,0.5351,0.6557,0.3987,0.6831,0.5399
DecisionTreeRegressor,0.5534,0.5696,0.7439,-0.0518,0.3846,0.3979,0.4084,0.4530,0.6391,0.4289,0.6705,0.7245
RandomForestRegressor,0.3556,0.4791,0.5963,0.3242,0.5694,0.5848,0.3866,0.4997,0.6218,0.4594,0.7322,0.5813
GradientBoostingRegressor,0.4495,0.4967,0.6705,0.1457,0.4491,0.4728,0.3378,0.4474,0.5812,0.5277,0.7756,0.6143
AdaBoostRegressor,0.4068,0.4995,0.6378,0.2268,0.5007,0.5129,0.3859,0.4918,0.6212,0.4604,0.7309,0.6061
XGBRegressor,0.4499,0.5205,0.6707,0.1450,0.4409,0.4682,0.4057,0.5144,0.6369,0.4327,0.6991,0.6143
ExtraTreesRegressor,0.3879,0.4981,0.6228,0.2627,0.5172,0.5538,0.4125,0.4901,0.6423,0.4232,0.6678,0.5289
LinearRegression,0.7219,0.6952,0.8496,-0.3720,0.4525,0.4587,0.4100,0.4836,0.6403,0.4267,0.7531,0.8375
KNeighborsRegressor,0.3293,0.4557,0.5739,0.3741,0.6175,0.6491,0.3770,0.5129,0.6140,0.4729,0.7319,0.5565
SVR,0.3558,0.4933,0.5965,0.3237,0.5725,0.5716,0.3359,0.4440,0.5796,0.5303,0.8209,0.7961


In [23]:
result_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/combined_features/combined_top_200_features_MDCK.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/combined_features/prediction_data_combined_top_200_features_MDCK.csv')

In [24]:
#Top 500 features
n = 500  
top_500_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_500_features = feature_names[top_500_indices].tolist()  # convert to list

# Output the list
print("Top", 500, "features:\n")
print(top_500_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_500_features]], axis=1)
test_df = test[train.columns]

Top 500 features:

['x_fine_emb_MFXL590', 'x_fine_emb_MFXL448', 'x_fine_emb_MFXL613', 'x_fine_emb_MFXL45', 'AATSC.17', 'x_fine_emb_MFXL226', 'x_fine_emb_MFXL386', 'x_fine_emb_MFXL12', 'x_fine_emb_MFXL146', 'x_fine_emb_MFXL366', 'x_fine_emb_MFXL514', 'DPSA-3', 'x_fine_emb_MFXL95', 'x_fine_emb_MFXL762', 'x_fine_emb_MFXL511', 'x_fine_emb_MFXL502', 'x_fine_emb_MFXL39', 'AtomTypeEState.187', 'x_fine_emb_MFXL125', 'AtomTypeEState.260', 'TDB7i', 'GATS.4', 'x_fine_emb_MFXL574', 'TDB9s', 'x_fine_emb_MFXL471', 'ATSC8e', 'x_fine_emb_MFXL766', 'x_fine_emb_MFXL434', 'x_fine_emb_MFXL684', 'x_fine_emb_MFXL566', 'x_fine_emb_MFXL249', 'AATS7i', 'x_fine_emb_MFXL721', 'x_fine_emb_MFXL504', 'x_fine_emb_MFXL751', 'x_fine_emb_MFXL184', 'x_fine_emb_MFXL305', 'x_fine_emb_MFXL673', 'TDB10m', 'x_fine_emb_MFXL147', 'maxHBint9', 'x_fine_emb_MFXL8', 'minHBint5', 'x_fine_emb_MFXL407', 'x_fine_emb_MFXL317', 'x_fine_emb_MFXL193', 'x_fine_emb_MFXL740', 'AATS.33', 'x_fine_emb_MFXL394', 'x_fine_emb_MFXL181', 'x_fine_emb

In [25]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 500)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 500)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.124920 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1749
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 117
[LightGBM] [Info] Start training from score -5.596983
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3422,0.4686,0.5850,0.3496,0.5959,0.6258,0.4587,0.5400,0.6773,0.3585,0.6427,0.4793
DecisionTreeRegressor,0.6052,0.5789,0.7780,-0.1503,0.3948,0.4517,0.3163,0.4458,0.5624,0.5577,0.7943,0.7080
RandomForestRegressor,0.3880,0.5027,0.6229,0.2626,0.5137,0.5509,0.4038,0.5191,0.6354,0.4354,0.7268,0.6088
GradientBoostingRegressor,0.4444,0.5180,0.6666,0.1554,0.4522,0.4662,0.3710,0.4732,0.6091,0.4812,0.7482,0.5592
AdaBoostRegressor,0.3834,0.4853,0.6192,0.2713,0.5290,0.5678,0.3822,0.4855,0.6182,0.4656,0.7369,0.6171
XGBRegressor,0.4607,0.5296,0.6788,0.1243,0.4328,0.4439,0.3848,0.5023,0.6203,0.4619,0.7307,0.6722
ExtraTreesRegressor,0.4062,0.5143,0.6373,0.2280,0.4832,0.5152,0.3986,0.4937,0.6314,0.4426,0.6949,0.5510
LinearRegression,0.6707,0.6661,0.8189,-0.2746,0.3755,0.3868,0.2994,0.4179,0.5472,0.5814,0.8073,0.8678
KNeighborsRegressor,0.3101,0.4367,0.5568,0.4107,0.6443,0.6869,0.3979,0.5144,0.6308,0.4436,0.6926,0.5614
SVR,0.3920,0.5194,0.6261,0.2550,0.5058,0.5135,0.3542,0.4645,0.5951,0.5048,0.8024,0.7631


In [26]:
result_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/combined_features/combined_top_500_features_MDCK.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/combined_features/prediction_data_combined_top_500_features_MDCK.csv')